<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/glasslego/ml-deep-learning-study/blob/main/src/deep_learning_example/01_titanic_survival_prediction.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />구글 코랩에서 실행하기</a>
  </td>
</table>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Simple neural network implementation without PyTorch
class SimpleNN:
    def __init__(self, input_size, hidden_sizes, output_size):
        self.layers = []
        sizes = [input_size] + hidden_sizes + [output_size]
        
        # Initialize weights and biases
        for i in range(len(sizes) - 1):
            self.layers.append({
                'W': np.random.randn(sizes[i], sizes[i+1]) * np.sqrt(2.0 / sizes[i]),
                'b': np.zeros((1, sizes[i+1])),
                'vW': np.zeros((sizes[i], sizes[i+1])),
                'vb': np.zeros((1, sizes[i+1]))
            })
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def sigmoid_derivative(self, x):
        return x * (1 - x)
    
    def relu(self, x):
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        return (x > 0).astype(float)
    
    def forward(self, X, training=True):
        self.activations = [X]
        self.z_values = []
        
        for i, layer in enumerate(self.layers):
            z = np.dot(self.activations[-1], layer['W']) + layer['b']
            self.z_values.append(z)
            
            # Use ReLU for hidden layers, sigmoid for output
            if i < len(self.layers) - 1:
                a = self.relu(z)
                if training:
                    # Dropout
                    mask = np.random.binomial(1, 0.8, size=a.shape) / 0.8
                    a *= mask
            else:
                a = self.sigmoid(z)
            
            self.activations.append(a)
        
        return self.activations[-1]
    
    def backward(self, X, y, lr=0.001, momentum=0.9, weight_decay=1e-5):
        m = X.shape[0]
        
        # Output layer error
        delta = self.activations[-1] - y
        
        # Backpropagate
        for i in range(len(self.layers) - 1, -1, -1):
            # Gradients
            dW = np.dot(self.activations[i].T, delta) / m
            db = np.sum(delta, axis=0, keepdims=True) / m
            
            # L2 regularization
            dW += weight_decay * self.layers[i]['W']
            
            # Momentum update
            self.layers[i]['vW'] = momentum * self.layers[i]['vW'] - lr * dW
            self.layers[i]['vb'] = momentum * self.layers[i]['vb'] - lr * db
            
            self.layers[i]['W'] += self.layers[i]['vW']
            self.layers[i]['b'] += self.layers[i]['vb']
            
            # Propagate error to previous layer
            if i > 0:
                delta = np.dot(delta, self.layers[i]['W'].T)
                delta *= self.relu_derivative(self.activations[i])
    
    def train(self, X, y, X_val, y_val, epochs=100, batch_size=32, lr=0.001, patience=15):
        history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(epochs):
            # Shuffle training data
            indices = np.random.permutation(len(X))
            X_shuffled = X[indices]
            y_shuffled = y[indices]
            
            # Mini-batch training
            for i in range(0, len(X), batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                self.forward(X_batch, training=True)
                self.backward(X_batch, y_batch, lr=lr)
            
            # Evaluate on training set
            y_pred_train = self.forward(X, training=False)
            train_loss = -np.mean(y * np.log(y_pred_train + 1e-7) + (1 - y) * np.log(1 - y_pred_train + 1e-7))
            train_acc = np.mean((y_pred_train > 0.5) == y)
            
            # Evaluate on validation set
            y_pred_val = self.forward(X_val, training=False)
            val_loss = -np.mean(y_val * np.log(y_pred_val + 1e-7) + (1 - y_val) * np.log(1 - y_pred_val + 1e-7))
            val_acc = np.mean((y_pred_val > 0.5) == y_val)
            
            history['loss'].append(train_loss)
            history['acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"\nEarly stopping at epoch {epoch+1}")
                    break
            
            # Learning rate decay
            if patience_counter > 5:
                lr *= 0.5
                patience_counter = 0
        
        return history
    
    def predict(self, X):
        return self.forward(X, training=False)


# 📊 Titanic Survival Prediction with Simple Neural Network


In [ ]:
# Load and prepare data
def prepare_titanic_data():
    """
    Prepare Titanic dataset
    """
    # Load sample Titanic data (since actual data path is not provided)
    # Creating a realistic sample dataset based on Titanic features
    np.random.seed(42)
    
    # Generate sample data with realistic Titanic features
    n_samples = 891
    
    # Features: Pclass, Sex, Age, SibSp, Parch, Fare, Embarked
    data = {
        'Pclass': np.random.choice([1, 2, 3], n_samples, p=[0.25, 0.2, 0.55]),
        'Sex': np.random.choice([0, 1], n_samples, p=[0.35, 0.65]),  # 0=female, 1=male
        'Age': np.random.normal(29.7, 14.5, n_samples),
        'SibSp': np.random.choice([0, 1, 2, 3, 4, 5], n_samples, p=[0.68, 0.23, 0.06, 0.02, 0.01, 0.005]),
        'Parch': np.random.choice([0, 1, 2, 3, 4, 5, 6], n_samples, p=[0.76, 0.13, 0.08, 0.02, 0.003, 0.003, 0.003]),
        'Fare': np.random.lognormal(2.8, 1.2, n_samples),
        'Embarked': np.random.choice([0, 1, 2], n_samples, p=[0.7, 0.2, 0.1])  # 0=S, 1=C, 2=Q
    }
    
    # Create survival target based on realistic patterns
    survival_prob = (
        (data['Sex'] == 0) * 0.75 +  # Females have higher survival rate
        (data['Pclass'] == 1) * 0.4 +  # First class has higher survival rate
        (data['Pclass'] == 2) * 0.2 +  # Second class moderate survival rate
        (data['Age'] < 15) * 0.3 +      # Children have higher survival rate
        (data['Fare'] > 50) * 0.2       # Higher fare indicates better survival
    )
    
    # Add some randomness
    survival_prob = np.clip(survival_prob + np.random.normal(0, 0.1, n_samples), 0, 1)
    survived = np.random.binomial(1, survival_prob, n_samples)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    df['Survived'] = survived
    
    # Handle missing values (simulate some missing ages)
    missing_age_idx = np.random.choice(n_samples, size=int(0.2 * n_samples), replace=False)
    df.loc[missing_age_idx, 'Age'] = df['Age'].median()
    
    return df

# Prepare data
print("📊 Titanic 데이터 준비 중...")
df = prepare_titanic_data()

print(f"데이터 형태: {df.shape}")
print(f"생존률: {df['Survived'].mean():.3f}")
print("\n데이터 샘플:")
print(df.head())

# Feature engineering
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X = df[features].values
y = df['Survived'].values

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Scale the features
scaler = StandardScaler()
X_train_split = scaler.fit_transform(X_train_split)
X_val_split = scaler.transform(X_val_split)
X_test = scaler.transform(X_test)

print(f"\n훈련 데이터: {X_train_split.shape}")
print(f"검증 데이터: {X_val_split.shape}")
print(f"테스트 데이터: {X_test.shape}")
print("✅ 데이터 준비 완료!")

In [3]:
# Training multiple models
models_config = {
    'Simple': [64],
    'Medium': [128, 64],
    'Deep': [128, 64, 32]
}

results = {}

# Assuming you have X_train_split, y_train_split, X_val_split, y_val_split defined
# Replace this with your actual data preparation

print("🚀 신경망 모델 훈련 시작\n")

for name, hidden_sizes in models_config.items():
    print(f"\n📊 {name} 모델 훈련 중...")
    
    # Create model (adjust input_size based on your features)
    input_size = X_train_split.shape[1]  # Number of features
    model = SimpleNN(input_size, hidden_sizes, output_size=1)
    
    # Train model
    history = model.train(
        X_train_split, y_train_split.reshape(-1, 1),
        X_val_split, y_val_split.reshape(-1, 1),
        epochs=100, batch_size=32, lr=0.001, patience=15
    )
    
    results[name] = {
        'model': model,
        'history': history,
        'best_val_acc': max(history['val_acc'])
    }
    
    print(f"✅ {name} 모델 완료 - 최고 검증 정확도: {results[name]['best_val_acc']:.4f}")

print("\n🎉 모든 모델 훈련 완료!")


🚀 신경망 모델 훈련 시작


📊 Simple 모델 훈련 중...


NameError: name 'X_train_split' is not defined

In [ ]:
# Visualize results
import matplotlib.pyplot as plt

# Check if results exist
if 'results' in locals() and results:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    for name, result in results.items():
        history = result['history']
        
        axes[0].plot(history['acc'], label=f'{name} Train')
        axes[0].plot(history['val_acc'], label=f'{name} Val', linestyle='--')
        
        axes[1].plot(history['loss'], label=f'{name} Train')
        axes[1].plot(history['val_loss'], label=f'{name} Val', linestyle='--')

    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

    # Test the best model
    print("\n🎯 테스트 데이터 평가:")
    best_model_name = max(results.keys(), key=lambda x: results[x]['best_val_acc'])
    best_model = results[best_model_name]['model']
    
    y_test_pred = best_model.predict(X_test)
    test_acc = np.mean((y_test_pred > 0.5).flatten() == y_test)
    
    print(f"최고 성능 모델: {best_model_name}")
    print(f"테스트 정확도: {test_acc:.4f}")
    
    # Confusion matrix
    from sklearn.metrics import confusion_matrix
    y_test_pred_binary = (y_test_pred > 0.5).astype(int).flatten()
    cm = confusion_matrix(y_test, y_test_pred_binary)
    
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(f'{best_model_name} Model - Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ['Not Survived', 'Survived'])
    plt.yticks(tick_marks, ['Not Survived', 'Survived'])
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    # Add text annotations
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ 먼저 모델을 훈련해주세요. 위의 셀을 실행하여 모델을 훈련한 후 다시 시도하세요.")

In [ ]:
# 필요한 라이브러리 import (Google Colab T4 환경)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# 나눔고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# 폰트 설정
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)  # 마이너스 기호 깨짐 방지

print(f"PyTorch 버전: {torch.__version__}")

# GPU 설정 (Google Colab T4 환경)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: {device}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 버전: {torch.version.cuda}")
else:
    device = torch.device("cpu")
    print(f"Using device: {device}")

# 시드 설정 (재현 가능한 결과를 위해)
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# 시각화 설정
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12